# Samatrix Practice: Retail Demand Forecasting

This notebook is designed to help you learn the ML workflow, not just produce a submission.

This guided ML lab helps you practice the full regression workflow.

You will inspect the dataset, choose useful features, train a baseline model, validate using RMSE, improve the model, create `predictions.csv`, and submit it to Samatrix Practice.

## Learning workflow

Observe -> Think -> Experiment -> Validate -> Improve -> Submit -> Reflect

The goal is to understand the ML process, not only generate a file.

## Step 1: Download the challenge files

Run the next cell first.

In [ ]:
#@title Step 1 : Download challenge files { display-mode: "form" }
#@markdown Run this cell once. You do not need to edit the code.

import json
import pathlib
from pathlib import Path

import pandas as pd
import requests

BASE_URL = "https://practice.samatrix.io"
CHALLENGE_SLUG = "retail-demand-001"
FORCE_DOWNLOAD = False

DATA_DIR = Path(".")
CONFIG_URL = f"{BASE_URL}/api/v1/challenges/{CHALLENGE_SLUG}/client-config"

print("Connecting to Samatrix Practice...")
response = requests.get(CONFIG_URL, timeout=30)

if response.status_code != 200:
    raise RuntimeError(f"Could not load challenge config: {response.status_code} {response.text[:300]}")

config = response.json()

files_to_download = {
    "train": "train.csv",
    "test": "test.csv",
    "sample_submission": "sample_submission.csv",
}

for asset_key, filename in files_to_download.items():
    path = DATA_DIR / filename

    if path.exists() and not FORCE_DOWNLOAD:
        print(f"Using existing {filename}")
        continue

    file_url = config["assets"][asset_key]
    file_response = requests.get(file_url, timeout=60)

    if file_response.status_code != 200:
        raise RuntimeError(f"Could not download {filename}: {file_response.status_code}")

    path.write_bytes(file_response.content)
    print(f"Downloaded {filename}")

print()
print("Challenge:", config.get("title"))
print("Metric:", config.get("task", {}).get("metric_name"))
print("Files are ready.")

## Step 2: Load the data

Run this cell to load:

- train.csv
- test.csv
- sample_submission.csv
- challenge configuration

In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")

print("train shape:", train.shape)
print("test shape:", test.shape)
print("sample_submission shape:", sample_submission.shape)

train.head()

## 3. Understand the dataset

Inspect:

- rows and columns
- target column
- ID column
- numeric features
- categorical features
- missing values
- target range

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

task = config.get("task", {})

target_column = task.get(
    "target_column",
    "units_sold"
)

id_column = task.get(
    "id_column",
    "id"
)

prediction_column = task.get(
    "prediction_column",
    "prediction"
)

print("Target:", target_column)
print("ID:", id_column)
print("Prediction column:", prediction_column)

print("Train:", train.shape)
print("Test:", test.shape)

display(train.head())

print("\nTarget summary:")
display(train[target_column].describe())

print("\nMissing values:")
display(train.isna().sum())

## Feature Selection

IDs identify rows. They usually should not be used as prediction features.

Start with the suggested features, then revise after inspecting the data.

In [ ]:
# TODO: Review candidate_features and remove any feature you believe is weak, noisy, duplicated, or risky.
candidate_features = [
    'store_type',
    'product_category',
    'region',
    'price',
    'competitor_price',
    'discount_percent',
    'ad_spend',
    'footfall',
    'is_weekend',
    'is_holiday',
    'local_event_score',
    'inventory_level',
    'temperature_c',
    'month'
]

feature_columns = candidate_features.copy()

print(
    feature_columns
)

## Checkpoint 1: Dataset observations

In [ ]:
# TODO: Replace the placeholder answers below with your own dataset observations.
checkpoint_1_dataset_observations = {
    "target_column": target_column,
    "target_pattern": "TODO: What did you notice about the target values?",
    "feature_pattern": "TODO: Which features look useful and why?",
    "possible_risk": "TODO: What could go wrong if we validate poorly?",
}

checkpoint_1_dataset_observations

## Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

X = train[feature_columns]
y = train[target_column]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))

## Encoding

In [ ]:
# TODO: Understand why one-hot encoding is needed before training the model.
X_train_model = pd.get_dummies(X_train, dummy_na=True)
X_valid_model = pd.get_dummies(X_valid, dummy_na=True)

X_valid_model = X_valid_model.reindex(
    columns=X_train_model.columns,
    fill_value=0
)

print(
    X_train_model.shape
)

## Baseline Model

## ML insight: RMSE

RMSE means Root Mean Squared Error.

For regression, predictions should be numeric target estimates, not probabilities or class labels.

Lower RMSE is better. Large errors are penalized more heavily because the errors are squared before averaging.

In [ ]:
# TODO: Confirm that this model predicts numeric values for the target column.
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

baseline_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=2,
    random_state=42,
)

baseline_model.fit(X_train_model, y_train)

valid_predictions = baseline_model.predict(X_valid_model)
valid_rmse = root_mean_squared_error(y_valid, valid_predictions)

print(
    "Validation RMSE:",
    round(valid_rmse, 4)
)

valid_score = valid_rmse

## Experiment

Try one change and record whether validation RMSE improved.

Examples:

- Add or remove features
- Change model depth
- Change number of trees
- Compare validation RMSE

In [ ]:
# TODO: Record at least one experiment and whether validation RMSE improved.
experiment_tracker = {
    "baseline_validation_score": valid_score,
    "change_tried": "TODO: Describe one change you tried.",
    "new_validation_score": "TODO: Record the new validation RMSE.",
    "decision": "TODO: Did you keep the change? Why?",
}

experiment_tracker

## Create predictions.csv

Important: If your experiment found better parameters or a better model, mirror your best configuration in the `final_model` cell below. Otherwise, you will submit the baseline configuration even if your experiment improved validation RMSE.

## Choose your final model configuration

Update this configuration using what you learned from validation experiments.

The final submission model will use `best_model_config`, so your best experiment naturally flows into `predictions.csv`.

In [ ]:
# TODO: Update this dictionary with your best validation experiment settings.
best_model_config = {
    "n_estimators": 100,
    "max_depth": 5,
    "min_samples_leaf": 2,
    "random_state": 42,
}

best_model_config

In [ ]:
X_full = train[feature_columns]
X_test = test[feature_columns]
y_full = train[target_column]

X_full_model = pd.get_dummies(X_full, dummy_na=True)
X_test_model = pd.get_dummies(X_test, dummy_na=True)

X_test_model = X_test_model.reindex(
    columns=X_full_model.columns,
    fill_value=0
)

final_model = RandomForestRegressor(**best_model_config)
final_model.fit(X_full_model, y_full)

test_predictions = final_model.predict(X_test_model)

submission = sample_submission.copy()
submission[prediction_column] = test_predictions

submission.to_csv(
    "predictions.csv",
    index=False
)

print("Created predictions.csv")
print("Submission rows:", len(submission))
display(submission.head())

## Reflection

Answer these before submitting. This helps you prepare for interviews and the Review Room.

In [ ]:
# TODO: Replace the placeholder answers below with your actual reflection before submitting.
student_reflection = {
    "what_worked": "TODO: What worked well in your model?",
    "what_you_changed": "TODO: What did you change after validation?",
    "what_you_would_try_next": "TODO: What would you try next if you had more time?",
}

student_reflection

## 4. Submit to Samatrix Practice

**Step 4: Run Cell, Paste Token, and Submit**

Before running the submit cell:

1. Generate a **Colab Submission Token** from the challenge page.
2. Copy the token.
3. Run the Step 4 cell below.
4. Paste the token when Colab asks for it.
5. Submit `predictions.csv`.
6. Open the **Direct Review Room** link to return to Samatrix Practice and review your result.

Do not paste your Samatrix username or password here. Only the Colab Submission Token is needed.

Colab may show a small **Show code** link for form cells. You can ignore it during normal use.

In [ ]:
#@title Step 4: Run Cell, Paste Token, and Submit { display-mode: "form" }

# Soft learning-evidence check. This does not block submission.
def _samatrix_todo_warning_contains(value):
    if isinstance(value, str):
        return "TODO" in value

    if isinstance(value, dict):
        return any(_samatrix_todo_warning_contains(v) for v in value.values())

    if isinstance(value, (list, tuple, set)):
        return any(_samatrix_todo_warning_contains(v) for v in value)

    try:
        import pandas as _samatrix_pd
        if isinstance(value, _samatrix_pd.DataFrame):
            return value.astype(str).apply(
                lambda column: column.str.contains("TODO", case=False, na=False)
            ).any().any()
    except Exception:
        pass

    return False

_samatrix_todo_warning_names = [
    "checkpoint_1_dataset_observations",
    "experiment_tracker",
    "student_reflection",
]

_samatrix_todo_warning_hits = [
    name
    for name in _samatrix_todo_warning_names
    if name in globals() and _samatrix_todo_warning_contains(globals()[name])
]

if _samatrix_todo_warning_hits:
    print("Learning reminder: Some sections still contain TODO.")
    print("Sections to review before submitting:", ", ".join(_samatrix_todo_warning_hits))
    print("Submission will continue, but replacing TODO notes will make your Review Room and mentor feedback more useful.")

import requests
from getpass import getpass
from IPython.display import display, HTML

if not pathlib.Path("predictions.csv").exists():
    raise FileNotFoundError("predictions.csv was not found. Run the prediction cell before submitting.")

token = getpass("Paste your Colab Submission Token: ").strip()

with open("predictions.csv", "rb") as file:
    response = requests.post(
        f"{BASE_URL}/api/v1/submissions/token",
        headers={"Authorization": f"Bearer {token}"},
        files={"predictions_file": ("predictions.csv", file, "text/csv")},
        timeout=60,
    )

if response.status_code >= 400:
    print("Submission failed.")
    print(response.text)
    response.raise_for_status()

result = response.json()
print("Submission successful.")
print("Practice score:", result.get("evaluation", {}).get("score_normalized"))
print("Review URL:", result.get("review_url"))

review_url = result.get("review_url")
if review_url:
    display(HTML(f'<a href="{review_url}" target="_blank">Open Direct Review Room</a>'))